# Causal Determinants of Body Weight: HUABB Health Study

**Dataset**: HUABB (BETTER4U consortium) — 14,203 participants from a multi-cohort European health study with 295 variables covering demographics, anthropometrics, dietary intake, physical activity, clinical biomarkers, and lifestyle factors.

**Research question**: What are the causal determinants of body weight and waist-to-hip ratio in a real-world health cohort?

**This notebook**:
1. Loads the expert-defined causal graph (373 edges, 87 nodes) and the HUABB dataset
2. Audits graph-data alignment — most graph nodes are 100% empty in the data
3. Prunes the graph to nodes with actual data, adds domain-justified edges
4. Imputes missing values using MICE (Multiple Imputation by Chained Equations)
5. Fits a CausalPype structural causal model

**Status**: Data loading, preprocessing, and model fitting only — causal experiments follow in a separate notebook.

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import warnings
warnings.filterwarnings("ignore")

import causalpype as cp

## 1. Load Data and Expert Causal Graph

In [ ]:
# Load the cleaned HUABB dataset
data = pd.read_csv("../HUABB_data_cleaned.csv", low_memory=False)
print(f"Dataset: {data.shape[0]:,} rows x {data.shape[1]} columns")

# Load the expert-defined causal edge list
# The CSV header encodes the first edge (column names = first source/target pair)
edges_raw = pd.read_csv("../edge_list - causal_graph.csv")
first_edge = pd.DataFrame({"source": [edges_raw.columns[0]], "target": [edges_raw.columns[1]]})
edges_raw.columns = ["source", "target"]
edges_df = pd.concat([first_edge, edges_raw], ignore_index=True)

print(f"Expert graph: {len(edges_df)} directed edges, "
      f"{len(set(edges_df['source']) | set(edges_df['target']))} unique nodes")

In [ ]:
# Audit graph-data alignment
graph_nodes = set(edges_df["source"]) | set(edges_df["target"])
data_cols = set(data.columns)

not_in_data = graph_nodes - data_cols
in_data = graph_nodes & data_cols
empty_nodes = {col for col in in_data if data[col].isna().all()}
has_data = in_data - empty_nodes

print(f"Graph nodes not in data at all:      {len(not_in_data):>3d}  -> {sorted(not_in_data)}")
print(f"Graph nodes in data but 100% empty:  {len(empty_nodes):>3d}")
print(f"Graph nodes with actual data:        {len(has_data):>3d}")

print(f"\nUsable nodes ({len(has_data)}) — sorted by missingness:")
for n in sorted(has_data, key=lambda c: data[c].isna().mean()):
    pct = data[n].isna().mean() * 100
    bar = "█" * int(pct / 2)
    print(f"  {n:35s} {pct:5.1f}%  {bar}")

## 2. Graph Pruning

The expert graph has 87 nodes but only 35 have any data at all. We prune systematically:

1. **Remove nodes not in data or 100% empty** (52 nodes) — entire variable categories (dietary percentages, detailed PA, health status) were never collected for this cohort
2. **Apply missingness threshold (60%)** — drops `sitting_day` (85.7%), `MET_week` (88.9%), `cigarettes_day` (61.6%), `lightbeverages_day` (70.7%), `age_quitting` (93.4%)
3. **Remove dead-end nodes** — `employment` only connected to the dropped `MET_week`/`PAL` nodes and `education_group`, becoming a leaf with no path to any outcome
4. **Add domain-justified edges** — `smoking_status -> bw` and `smoking_status -> whr` (smoking is a well-established determinant of body weight and fat distribution; the original graph had smoking edges to nodes that were 100% empty)

Note: `PAL` (Physical Activity Level, 59.4% missing) is kept as the sole physical activity indicator since it's just below the threshold and is an important body weight determinant.

In [ ]:
# Step 1: Keep only edges where both endpoints have actual data
usable_edges = edges_df[
    edges_df["source"].isin(has_data) & edges_df["target"].isin(has_data)
].copy()
print(f"After removing empty/missing nodes: {len(usable_edges)} edges")

# Step 2: Apply missingness threshold — drop nodes with >60% missing
MISS_THRESHOLD = 0.60
high_missing = {col for col in has_data if data[col].isna().mean() > MISS_THRESHOLD}
print(f"Dropped (>{MISS_THRESHOLD:.0%} missing): {sorted(high_missing)}")

usable_edges = usable_edges[
    ~usable_edges["source"].isin(high_missing) & ~usable_edges["target"].isin(high_missing)
]

# Step 3: Remove employment (dead-end leaf — its targets MET_week/PAL were dropped)
usable_edges = usable_edges[
    (usable_edges["source"] != "employment") & (usable_edges["target"] != "employment")
]

# Step 4: Add domain-justified smoking edges
smoking_edges = pd.DataFrame({
    "source": ["smoking_status", "smoking_status"],
    "target": ["bw", "whr"]
})
usable_edges = pd.concat([usable_edges, smoking_edges], ignore_index=True)

# Build adjacency dict for CausalPype
from collections import defaultdict
adj = defaultdict(list)
for _, row in usable_edges.iterrows():
    adj[row["source"]].append(row["target"])

# Verify DAG
G = nx.DiGraph(dict(adj))
assert nx.is_directed_acyclic_graph(G), f"Cycle detected: {list(nx.find_cycle(G))}"

roots = [n for n in G if G.in_degree(n) == 0]
leaves = [n for n in G if G.out_degree(n) == 0]
print(f"\nFinal graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Root nodes (exogenous):  {sorted(roots)}")
print(f"Leaf nodes (outcomes):   {sorted(leaves)}")
print(f"\nEdge list:")
for src in sorted(adj.keys()):
    if adj[src]:
        print(f"  {src} -> {sorted(adj[src])}")

## 3. Data Preprocessing

### Variable types
- **Continuous** (8): `age`, `bh` (body height), `bw` (body weight), `whr` (waist-to-hip ratio), `energy_intake_day`, `sleep_hours`, `sugarsweetenedbeverages_day`, `PAL` (physical activity level)
- **Categorical** (4): `sex` (1=male, 2=female), `education_group` (1, 2, 4), `social_integration` (0/1), `smoking_status` (0/1)

### Imputation strategy
1. **Categoricals** — mode imputation (most frequent category)
2. **Continuous** — MICE (`IterativeImputer` with `BayesianRidge`) using the already-imputed categoricals as auxiliary predictors

MICE preserves the correlation structure between variables, which is critical for causal modeling — simple mean/median imputation would break the very relationships we are trying to model.

In [ ]:
# Select only the columns needed for the causal graph
graph_cols = sorted(G.nodes)
df = data[graph_cols].copy()

print(f"Selected {len(graph_cols)} columns, {len(df):,} rows\n")
print("Missingness summary:")
missing = df.isna().mean().sort_values(ascending=False) * 100
for col, pct in missing.items():
    bar = "█" * int(pct / 2)
    print(f"  {col:35s} {pct:5.1f}%  {bar}")

complete = df.dropna().shape[0]
print(f"\nComplete cases: {complete:,} / {len(df):,} ({complete/len(df):.1%})")

In [ ]:
# Inspect categorical variables before imputation
cat_cols = ["sex", "education_group", "social_integration", "smoking_status"]
cont_cols = [c for c in graph_cols if c not in cat_cols]

for col in cat_cols:
    vals = df[col].dropna()
    print(f"{col}: {vals.nunique()} categories, values={sorted(vals.unique())}, "
          f"missing={df[col].isna().sum()} ({df[col].isna().mean():.1%})")
    print(f"  {vals.value_counts().sort_index().to_dict()}\n")

In [ ]:
# Phase 1: Impute categoricals with mode (most frequent category)
from sklearn.impute import SimpleImputer

cat_imputer = SimpleImputer(strategy="most_frequent")
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

# Convert to proper categorical dtype
# DoWhy's auto.assign_causal_mechanisms checks dtype to assign ClassifierFCM vs regressors
for col in cat_cols:
    df[col] = df[col].astype(int).astype("category")

print("Categorical imputation complete:")
for col in cat_cols:
    print(f"  {col}: dtype={df[col].dtype}, categories={df[col].cat.categories.tolist()}, "
          f"missing={df[col].isna().sum()}")

In [ ]:
# Phase 2: Impute continuous variables with MICE (IterativeImputer)
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer

print(f"Continuous columns ({len(cont_cols)}): {cont_cols}")
print(f"\nMissing before imputation:")
for c in cont_cols:
    print(f"  {c:35s} {df[c].isna().sum():>5d} ({df[c].isna().mean():.1%})")

# Build numeric matrix (encode categoricals as integers for the imputer)
df_for_mice = df.copy()
for col in cat_cols:
    df_for_mice[col] = df_for_mice[col].cat.codes.astype(float)

# MICE imputation with BayesianRidge (default estimator, good for mixed data)
mice = IterativeImputer(max_iter=20, random_state=42, sample_posterior=False)
imputed_values = pd.DataFrame(
    mice.fit_transform(df_for_mice),
    columns=df_for_mice.columns,
    index=df_for_mice.index,
)

# Write back only the continuous columns (categoricals already imputed)
for col in cont_cols:
    df[col] = imputed_values[col].values

# Restore categorical dtypes
for col in cat_cols:
    df[col] = df[col].astype("category")

assert df.isna().sum().sum() == 0, "Still have missing values!"
print(f"\nImputation complete. Zero missing values confirmed.")
print(f"Final shape: {df.shape}")

In [ ]:
# Post-imputation sanity checks
print("Continuous variable statistics after imputation:")
print(df[cont_cols].describe().round(2).to_string())

# Flag any implausible values
print("\n\nPlausibility checks:")
checks = {
    "age": (0, 120, "years"),
    "bh": (50, 250, "cm"),
    "bw": (10, 300, "kg"),
    "whr": (0.4, 1.5, "ratio"),
    "energy_intake_day": (0, 10000, "kcal"),
    "sleep_hours": (0, 24, "hours"),
    "sugarsweetenedbeverages_day": (0, None, "g"),
    "PAL": (0, 5, "ratio"),
}
for col, (lo, hi, unit) in checks.items():
    vals = df[col]
    issues = []
    if lo is not None and (vals < lo).any():
        issues.append(f"{(vals < lo).sum()} below {lo}")
    if hi is not None and (vals > hi).any():
        issues.append(f"{(vals > hi).sum()} above {hi}")
    status = ", ".join(issues) if issues else "OK"
    print(f"  {col:35s} [{lo}-{hi} {unit}]: {status}")

print("\nCategorical distributions:")
for col in cat_cols:
    print(f"  {col}: {df[col].value_counts().sort_index().to_dict()}")

## 4. Build and Fit the Causal Model

The pruned DAG encodes the following causal structure:

**Exogenous (root) nodes**: age, sex, education_group, energy_intake_day, sleep_hours, social_integration, sugarsweetenedbeverages_day, smoking_status

**Mediators**: bh (body height), PAL (physical activity level), sleep_hours

**Outcome (leaf) nodes**: bw (body weight), whr (waist-to-hip ratio)

**Key causal pathways to body weight**:
- **Demographics**: age -> bw, sex -> bw, education_group -> bw
- **Anthropometric**: sex -> bh -> bw (height mediates sex's effect on weight)
- **Physical activity**: age/sex -> PAL -> bw
- **Nutrition**: energy_intake_day -> bw, sugarsweetenedbeverages_day -> bw
- **Lifestyle**: sleep_hours -> bw, social_integration -> bw, smoking_status -> bw

In [ ]:
# Define the pruned causal graph
graph = {
    "age":                          ["sleep_hours", "bw", "PAL", "whr"],
    "sex":                          ["bh", "bw", "PAL", "whr"],
    "bh":                           ["bw"],
    "PAL":                          ["bw"],
    "education_group":              ["bw"],
    "energy_intake_day":            ["bw", "whr"],
    "sleep_hours":                  ["bw"],
    "social_integration":           ["bw"],
    "sugarsweetenedbeverages_day":  ["bw", "whr"],
    "smoking_status":               ["bw", "whr"],
}

model = cp.CausalModel(graph)
print(model)
print(f"\nRoot nodes:  {model.get_roots()}")

In [ ]:
%%time
model.fit(df)
print(model)

In [ ]:
# Verify assigned causal mechanisms
print("Assigned causal mechanisms:")
for node in sorted(model.graph.nodes):
    mechanism = model.scm.causal_mechanism(node)
    mtype = type(mechanism).__name__
    print(f"  {node:35s} {mtype}")

# Quick sanity check: synthetic samples vs observed
synthetic = model.draw_samples(n=2000)
print("\nSynthetic vs. observed means (continuous):")
for col in cont_cols:
    obs_mean = df[col].mean()
    syn_mean = synthetic[col].mean()
    print(f"  {col:35s} observed={obs_mean:8.2f}  synthetic={syn_mean:8.2f}")

## Summary

**What we built**: A structural causal model over 14,203 HUABB participants, with 12 nodes and 19 directed edges centered on body weight (`bw`) and waist-to-hip ratio (`whr`).

**Graph pruning**: The expert-defined 87-node graph was reduced to 12 nodes because:
- 52 nodes had no data or were 100% empty (dietary percentages, detailed physical activity, health status indicators)
- 5 nodes dropped for >60% missing (sitting_day, MET_week, cigarettes_day, lightbeverages_day, age_quitting)
- 1 node (employment) became a dead-end after its targets were removed
- 2 domain-justified edges added: smoking_status -> {bw, whr}
- PAL (59.4% missing) retained as the sole physical activity indicator

**Imputation**: Mode for 4 categoricals, MICE (IterativeImputer, 20 iterations) for 8 continuous variables. This preserves the correlation structure needed for causal modeling.

**Next steps** (separate notebook):
- Causal effect estimation: ATE/CATE of dietary intake on body weight
- Dose-response curves for energy intake and SSBs
- Arrow strength and intrinsic causal influence decomposition
- Counterfactual analysis and model validation